# 03. Phase 1 Results Analysis

Post-hoc analysis notebook для уже обученных encoder-baselines.
По умолчанию использует `artifacts/summary/` и, если доступен полный архив результатов,
может опираться на распакованный `phase1_baselines`.

In [ ]:
%pip install -q transformers sentencepiece scikit-learn seaborn plotly bertviz

In [ ]:
import json
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif (PROJECT_ROOT / "research_baselines").exists():
    pass

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

sns.set_theme(style="whitegrid")

SUMMARY_DIR = PROJECT_ROOT / "artifacts" / "summary"
RESULTS_ROOT_CANDIDATES = [
    PROJECT_ROOT / "outputs" / "phase1_baselines",
    PROJECT_ROOT / "release_assets" / "results" / "phase1_baselines",
    PROJECT_ROOT / "phase1_baselines",
]

RESULTS_ROOT = next((path for path in RESULTS_ROOT_CANDIDATES if path.exists()), None)
RAW_DATASET_PATH = PROJECT_ROOT / "data" / "cocolofa_ru_v2.jsonl"
MASKED_DATASET_PATH = PROJECT_ROOT / "data" / "cocolofa_ru_v2_masked.jsonl"

{
    "summary_dir": str(SUMMARY_DIR),
    "results_root": str(RESULTS_ROOT) if RESULTS_ROOT else None,
    "raw_dataset_path": str(RAW_DATASET_PATH),
    "masked_dataset_path": str(MASKED_DATASET_PATH),
}

In [ ]:
summary = json.loads((SUMMARY_DIR / "summary_metrics.json").read_text(encoding="utf-8"))
completed = json.loads((SUMMARY_DIR / "completed_runs.json").read_text(encoding="utf-8"))

summary_df = pd.DataFrame(
    [
        {
            "group": group_key,
            "model_name": payload["model_name"],
            "text_column": payload["text_column"],
            "seed_count": payload["seed_count"],
            "macro_f1_mean": payload["macro_f1"]["mean"],
            "macro_f1_std": payload["macro_f1"]["std"],
            "macro_precision_mean": payload["macro_precision"]["mean"],
            "macro_recall_mean": payload["macro_recall"]["mean"],
            "accuracy_mean": payload["accuracy"]["mean"],
        }
        for group_key, payload in summary["groups"].items()
    ]
).sort_values("macro_f1_mean", ascending=False)

display(summary_df)

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=summary_df, x="group", y="macro_f1_mean", color="#2e8b57")
plt.xticks(rotation=20, ha="right")
plt.title("Macro-F1 by Encoder Configuration")
plt.ylabel("Macro-F1 mean")
plt.xlabel("Configuration")
plt.tight_layout()
plt.show()

In [ ]:
if RESULTS_ROOT and RESULTS_ROOT.exists():
    print("Full run directories detected:", RESULTS_ROOT)
    print("You can use this notebook together with docs/interactive/attention/*.html and outputs from the full archive.")
else:
    print("Full run directories are not available. Summary-level analysis still works from artifacts/summary/.")

## Interactive artifacts

Если включён GitHub Pages from `/docs`, интерактивные HTML доступны через:

- `docs/interactive/attention/bertviz_head_view_slippery_slope.html`
- `docs/interactive/attention/bertviz_model_view_slippery_slope.html`
- `docs/interactive/attention/bertviz_head_view_appeal_to_authority.html`
- `docs/interactive/attention/bertviz_model_view_appeal_to_authority.html`